In [0]:
from pyspark.sql.functions import lit
from pyspark.sql import DataFrame
from functools import reduce

catalog = 'operations'
schema = 'finance_staging'

volume_base = "/Volumes/operations/finance_staging/edgar_data"

years = ['2020', '2021', '2022', '2023', '2024', '2025']
quarters = ['q1', 'q2', 'q3', 'q4']
files = ['pre', 'num', 'sub', 'tag']

for file in files:
    partition_dfs = []

    for year in years:
        for quarter in quarters:
            filing_path = f"{volume_base}/{year}/{quarter}/{file}.txt"

            filing = (
                spark.read
                .option("sep", "\t")
                .option("header", "true")
                .option("inferSchema", "true")
                .option("nullValue", "")
                .csv(filing_path)
                .withColumn("source_file", lit(filing_path))
                .withColumn("source_file_description", lit(f"{year}_{quarter}_{file}"))
            )

            partition_dfs.append(filing)

    # Union all year/quarter partitions into one DataFrame
    combined = reduce(DataFrame.union, partition_dfs)

    # Write one table per file type
    (
        combined.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f'{catalog}.{schema}.raw_{file}_tbl')
    )

    print(f"Written: {catalog}.{schema}.raw_{file}_tbl")